In [7]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# Silver Layer

## Objective

Transform the raw Bronze dataset into a clean, validated and standardised Delta table suitable for analytical workloads.

The Silver layer applies data-quality rules, creates derived business attributes and prepares the data for downstream Gold reporting.

StatementMeta(, c8923f51-9429-4c7b-9bfb-d97f6378549d, 9, Finished, Available, Finished, False)

SyntaxError: invalid syntax (447079170.py, line 7)

In [ ]:
from pyspark.sql import functions as F

raw_df = spark.read.parquet(
    "Files/green_tripdata_2026-01.parquet"
)

In [ ]:
raw_df.printSchema()

raw_df.count()

In [6]:
silver_df = raw_df.dropDuplicates()

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 8, Finished, Available, Finished, False)

In [5]:
from pyspark.sql import functions as F

silver_df = silver_df.filter(
    F.col("VendorID").isNotNull()
)

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 7, Finished, Available, Finished, False)

NameError: name 'silver_df' is not defined

In [9]:
silver_df = silver_df.filter(
    F.col("trip_distance") > 0
)

StatementMeta(, 1aa4715f-3f2f-4fac-a371-e02c21a5d1c7, 11, Finished, Available, Finished, False)

In [11]:
silver_df = silver_df.filter(
    F.col("fare_amount") >= 0
)

StatementMeta(, 1aa4715f-3f2f-4fac-a371-e02c21a5d1c7, 13, Finished, Available, Finished, False)

In [14]:
silver_df = silver_df.filter(
    F.col("lpep_pickup_datetime")
    
    F.col("lpep_dropoff_datetime")
)

StatementMeta(, 1aa4715f-3f2f-4fac-a371-e02c21a5d1c7, 16, Finished, Available, Finished, False)

In [8]:
silver_df = silver_df.withColumn(
    "pickup_date",
    F.to_date("lpep_pickup_datetime")
)

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 10, Finished, Available, Finished, False)

In [10]:
silver_df = silver_df.withColumn(
    "pickup_hour",
    F.hour("lpep_pickup_datetime")
)

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 12, Finished, Available, Finished, False)

In [12]:
silver_df = silver_df.withColumn(
    "trip_duration_minutes",
    (
        F.unix_timestamp("lpep_dropoff_datetime")
        -
        F.unix_timestamp("lpep_pickup_datetime")
    ) / 60
)

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 14, Finished, Available, Finished, False)

In [14]:
silver_df = silver_df.withColumn(
    "revenue_per_mile",
    F.col("total_amount") / F.col("trip_distance")
)

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 16, Finished, Available, Finished, False)

In [16]:
from pyspark.sql import functions as F

raw_df = spark.read.parquet(
    "Files/green_tripdata_2026-01.parquet"
)

silver_df = (
    raw_df
    .dropDuplicates()
    .filter(F.col("VendorID").isNotNull())
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("fare_amount") >= 0)
    .filter(F.col("total_amount") >= 0)
    .filter(
        F.col("lpep_pickup_datetime")
        < F.col("lpep_dropoff_datetime")
    )
)

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 18, Finished, Available, Finished, False)

In [19]:
silver_df = (
    silver_df
    .withColumn(
        "pickup_date",
        F.to_date("lpep_pickup_datetime")
    )
    .withColumn(
        "pickup_hour",
        F.hour("lpep_pickup_datetime")
    )
    .withColumn(
        "trip_duration_minutes",
        (
            F.unix_timestamp("lpep_dropoff_datetime")
            - F.unix_timestamp("lpep_pickup_datetime")
        ) / 60
    )
    .withColumn(
        "revenue_per_mile",
        F.col("total_amount") / F.col("trip_distance")
    )
)

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 21, Finished, Available, Finished, False)

In [21]:
display(
    silver_df.select(
        "lpep_pickup_datetime",
        "lpep_dropoff_datetime",
        "pickup_date",
        "pickup_hour",
        "trip_duration_minutes",
        "revenue_per_mile"
    ).limit(20)
)

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3a89b396-1956-4831-b948-9bf4f161dd39)

In [23]:
 bronze_count = raw_df.count()
silver_count = silver_df.count()

print(f"Bronze rows: {bronze_count}")
print(f"Silver rows: {silver_count}")
print(f"Rows removed: {bronze_count - silver_count}")


StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 25, Finished, Available, Finished, False)

Bronze rows: 40272
Silver rows: 38909
Rows removed: 1363


In [25]:
validation_df = silver_df.select(
    F.sum(F.col("VendorID").isNull().cast("int")).alias("NullVendorID"),
    F.sum((F.col("trip_distance") <= 0).cast("int")).alias("InvalidDistance"),
    F.sum((F.col("fare_amount") < 0).cast("int")).alias("NegativeFare"),
    F.sum((F.col("total_amount") < 0).cast("int")).alias("NegativeTotal"),
    F.sum(
        (
            F.col("lpep_pickup_datetime")
            >= F.col("lpep_dropoff_datetime")
        ).cast("int")
    ).alias("InvalidTimestamps")
)

display(validation_df)

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0635f4ee-7f2a-44ba-a836-fa4d21dfe7dc)

In [27]:
display(
    silver_df.select(
        "VendorID",
        "pickup_date",
        "pickup_hour",
        "trip_duration_minutes",
        "revenue_per_mile"
    ).limit(20)
)

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 91355106-6f47-4df6-85a1-8e1311e83ce7)

In [30]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_green_taxi_trips")
    

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 32, Finished, Available, Finished, False)

In [32]:
%%sql
SELECT COUNT(*) AS SilverTrips
FROM silver_green_taxi_trips;

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 34, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [1]:
# Silver Layer Summary

## Objective

#Transform the Bronze dataset into a clean, validated and standardised analytical dataset.

## Transformations Applied

#- Removed duplicate records.
#- Removed records with missing VendorID values.
#- Removed invalid trip distances.
#- Removed negative fare values.
#- Validated pickup and drop-off timestamps.
#- Added business-friendly calculated columns:
    - pickup_date
    - pickup_hour
    - trip_duration_minutes
    - revenue_per_mile

## Output

#The transformed dataset has been stored as the Delta table:

#silver_green_taxi_trips

#This dataset is now suitable for business modelling within the Gold layer.


StatementMeta(, fa06e9c1-f350-40ae-8843-f79f1cca6d92, 3, Finished, Available, Finished, False)

SyntaxError: invalid syntax (3359619001.py, line 5)

In [3]:
silver_df = spark.table( "silver_green_taxi_trips")




StatementMeta(, c8923f51-9429-4c7b-9bfb-d97f6378549d, 5, Finished, Available, Finished, False)

In [12]:
from pyspark.sql import functions as F

silver_df = spark.table("silver_green_taxi_trips")

display(silver_df.limit(10))

StatementMeta(, fa06e9c1-f350-40ae-8843-f79f1cca6d92, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a8ac9f70-0db4-4be7-b954-cef426eefc4f)

In [5]:
gold_vendor = (
    silver_df
    .groupBy("VendorID")
    .agg(
        ...
    )
)

StatementMeta(, 348927da-f7af-45db-9596-0987a499e513, 7, Finished, Available, Finished, False)

NameError: name 'silver_df' is not defined

In [6]:
from pyspark.sql import functions as F

gold_vendor = (
    silver_df
    .groupBy("VendorID")
    .agg(
        F.count("*").alias("TotalTrips"),
        F.avg("trip_distance").alias("AverageTripDistance"),
        F.avg("fare_amount").alias("AverageFare"),
        F.sum("total_amount").alias("TotalRevenue"),
        F.avg("revenue_per_mile").alias("AverageRevenuePerMile")
    )
)

StatementMeta(, c8923f51-9429-4c7b-9bfb-d97f6378549d, 8, Finished, Available, Finished, False)

In [13]:
display(gold_vendor)

StatementMeta(, 348927da-f7af-45db-9596-0987a499e513, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5d92477c-07de-4f00-9b54-5171cc7aeba1)

In [15]:


gold_vendor.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_vendor_performance")

StatementMeta(, 348927da-f7af-45db-9596-0987a499e513, 19, Finished, Available, Finished, False)

In [18]:
gold_daily = (
    silver_df
    .groupBy("pickup_date")
    .agg(
        F.count("*").alias("TotalTrips"),
        F.round(F.sum("total_amount"), 2).alias("TotalRevenue"),
        F.round(F.avg("fare_amount"), 2).alias("AverageFare"),
        F.round(F.avg("trip_distance"), 2).alias("AverageTripDistance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("AverageTripDurationMinutes")
    )
    .orderBy("pickup_date")
)

StatementMeta(, 348927da-f7af-45db-9596-0987a499e513, 22, Finished, Available, Finished, False)

In [20]:
display(gold_daily)

StatementMeta(, 348927da-f7af-45db-9596-0987a499e513, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 78cf205f-2dfc-4f4a-8d59-26490b253928)

In [22]:
gold_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_daily_trip_summary")

StatementMeta(, 348927da-f7af-45db-9596-0987a499e513, 26, Finished, Available, Finished, False)

In [10]:
gold_fare = (
    silver_df
    .groupBy("VendorID")
    .agg(
        F.round(F.avg("fare_amount"), 2).alias("AverageFare"),
        F.round(F.avg("tip_amount"), 2).alias("AverageTip"),
        F.round(F.avg("total_amount"), 2).alias("AverageTotalAmount"),
        F.round(F.avg("revenue_per_mile"), 2).alias("AverageRevenuePerMile")
    )
)

StatementMeta(, c8923f51-9429-4c7b-9bfb-d97f6378549d, 12, Finished, Available, Finished, False)

In [14]:
gold_fare.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_fare_analysis")


StatementMeta(, c8923f51-9429-4c7b-9bfb-d97f6378549d, 16, Finished, Available, Finished, False)

In [19]:
for table in [
    "gold_vendor_performance",
    "gold_daily_trip_summary",
    "gold_hourly_demand",
    "gold_fare_analysis"
]:
    print(table, spark.table(table).count())

StatementMeta(, c8923f51-9429-4c7b-9bfb-d97f6378549d, 21, Finished, Available, Finished, False)

gold_vendor_performance 3
gold_daily_trip_summary 33
gold_hourly_demand 24
gold_fare_analysis 3


In [20]:
gold_hourly = (
    silver_df
    .groupBy("pickup_hour")
    .agg(
        F.count("*").alias("TotalTrips"),
        F.round(F.sum("total_amount"), 2).alias("TotalRevenue"),
        F.round(F.avg("fare_amount"), 2).alias("AverageFare"),
        F.round(F.avg("trip_distance"), 2).alias("AverageTripDistance")
    )
    .orderBy("pickup_hour")
)

display(gold_hourly)



gold_hourly.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_hourly_demand")

StatementMeta(, c8923f51-9429-4c7b-9bfb-d97f6378549d, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c44b1c3d-bf50-4290-a985-179c97f8a2cf)

In [22]:
display(spark.table("gold_vendor_performance"))
display(spark.table("gold_daily_trip_summary").limit(10))
display(spark.table("gold_hourly_demand"))

StatementMeta(, c8923f51-9429-4c7b-9bfb-d97f6378549d, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6eac76f2-4611-4c7c-b57a-169dc12eb778)

SynapseWidget(Synapse.DataFrame, 25e7f927-0f4a-47c5-9632-9b2d7298c53f)

SynapseWidget(Synapse.DataFrame, d162ce8e-d76f-481a-9ecb-07366711144e)

In [ ]:
# Gold Layer Summary

## Objective

#Create business-ready analytical datasets from the validated Silver layer.

## Gold Tables Created

#- gold_vendor_performance
#- gold_daily_trip_summary
#- gold_hourly_demand
#- gold_fare_analysis

## Business Use

#These tables provide reusable metrics for vendor performance, daily trends, hourly demand and fare analysis.

#The Gold layer is now ready to support T-SQL reporting and Power BI.